In [38]:
#imports
import polars as pl
import json

import requests
from deepface import DeepFace
from tqdm import tqdm


In [39]:
session_path = r'C:\Users\Porter\Desktop\jcp\merge_data\jcpst-sessions-2026-04-07-22-48-30.json'


In [40]:
linkedin_path=r'C:\Users\Porter\Desktop\jcp\merge_data\linkedin-member-data-2026-04-07-224846.json'

In [41]:
import json

with open(linkedin_path, "r", encoding="utf-8") as f:
    obj = json.load(f)

linkedin = pl.DataFrame(obj)



In [42]:
import json
import polars as pl

path = r'C:\Users\Porter\Desktop\jcp\merge_data\jcpst-sessions-2026-04-07-22-48-30.json'

with open(path, "r", encoding="utf-8") as f:
    obj = json.load(f)

sessions = pl.DataFrame(obj["sessions"])



In [43]:

#for the linkedin cols
def expand_struct(df, colname):
    return df.with_columns(
        pl.col(colname).struct.unnest()
    )

#for treatment cols
def expand_list_struct_drop_inner(df: pl.DataFrame, col: str) -> pl.DataFrame:
    exploded = df.explode(col)

    inner_names = [f.name for f in exploded.schema[col].fields]
    outer_names = set(exploded.columns) - {col}

    keep_inner = [name for name in inner_names if name not in outer_names]

    return (
        exploded
        .with_columns(
            pl.struct(
                [pl.col(col).struct.field(name).alias(name) for name in keep_inner]
            ).alias(col)
        )
        .unnest(col)
    )

def merge_user_data(linkedin,sessions):
    #load the sessions json
    with open(sessions, "r", encoding="utf-8") as f:
        obj = json.load(f)

    sessions_df = pl.DataFrame(obj["sessions"])
    #load the linkedin json
    with open(linkedin, "r", encoding="utf-8") as f:
        obj = json.load(f)

    linkedin_df = pl.DataFrame(obj)
     #rename user-id so it matches the session schema, also rename other vars for clarity
    linkedin_df=linkedin_df.rename({'wordpress_user_id':'user_id',"last_synced_at":'account_last_synced_at','token_expires_at':'linkedin_token_expires_at','ip_history':'linkedin_ip_history','profile_data':'linkedin_profile_data'})
    linkedin_df=linkedin_df.with_columns(pl.col('user_id').cast(pl.Utf8).alias('user_id')) #cast to a string
    linkedin_df=linkedin_df.select(pl.exclude('wordpress_display_name')) #this column is redundant
    

    merged=sessions_df.join(linkedin_df,on=['user_id'],how='left')

    #expand the struct cols
    merged_expand = expand_struct(merged, "linkedin_profile_data")
    merged_expand = expand_list_struct_drop_inner(merged_expand, "treatment_snapshot")

    #make a unique key
    merged_expand = merged_expand.with_columns((pl.col("session_id") + "-" + pl.col("survey_id").fill_null('')).alias("session_survey_id"))

    # assert if it is a unique key
    assert merged_expand.select(pl.col("session_survey_id").is_unique().all()).item()

    return merged_expand

In [44]:
merged=merge_user_data(linkedin=linkedin_path,sessions=session_path)
merged

session_id,user_id,user_display_name,session_start,last_activity,session_end,duration_seconds,total_pageviews,visited_pages,ip,date,survey_id,treatment_group,post_url,job_ad_url,first_referrer,first_ip,last_ip,ip_hash,user_agent,device_summary,is_logged_in,login_state_changed,created_at,updated_at,record_id,linkedin_member_id,email,account_last_synced_at,linkedin_token_expires_at,linkedin_ip_history,linkedin_profile_data,sub,email_verified,name,locale,given_name,family_name,picture,session_survey_id
str,str,str,str,str,str,i64,i64,list[struct[2]],str,str,str,i64,str,str,str,str,str,str,str,str,bool,bool,str,str,i64,str,str,str,str,list[str],struct[8],str,bool,str,struct[2],str,str,str,str
"""3aa8ba57db3a60757ddc1e1e875b3c…",null,"""""","""2026-04-07 22:47:17""","""2026-04-07 22:47:17""",null,0,0,[],null,null,null,null,null,null,"""""","""192.0.84.117""","""192.0.84.117""","""55b50e648d132e75de60b15ae9e108…","""jetmon/1.0 (Jetpack Site Uptim…","""Desktop / Unknown Browser / Un…",false,false,"""2026-04-07 22:47:17""","""2026-04-07 22:47:17""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""3aa8ba57db3a60757ddc1e1e875b3c…"
"""0433769e922c7b83be73156ce3ad77…","""6""","""Porter Olson""","""2026-04-07 22:42:20""","""2026-04-07 22:43:59""",null,99,5,"[{""2026-04-07 22:42:20"",""https://jobconnectionsproject.org/""}, {""2026-04-07 22:42:28"",""https://jobconnectionsproject.org/jobs/""}, … {""2026-04-07 22:43:57"",""https://jobconnectionsproject.org/jobs/pastry-chef-san-jose-ca-us/""}]","""128.187.116.26""","""26-04-07 10:42:44""","""5256633647""",0,"""https://jobconnectionsproject.…","""https://jobconnectionsproject.…","""https://www.linkedin.com/""","""128.187.116.26""","""128.187.116.26""","""4fc992883f07c878a88e2feaff047f…","""Mozilla/5.0 (Windows NT 10.0; …","""Desktop / Chrome / Windows""",true,false,"""2026-04-07 22:42:20""","""2026-04-07 22:43:59""",1,"""AKLfR3WKvk""","""porter.olson11@gmail.com""","""2026-04-07 22:42:20""","""2026-06-06 22:42:19""","[""128.187.116.26"", ""67.169.249.168""]","{""AKLfR3WKvk"",true,""Porter Olson"",{""US"",""en""},""Porter"",""Olson"",""porter.olson11@gmail.com"",""https://media.licdn.com/dms/image/v2/D5603AQG9oDGMQr7JTw/profile-displayphoto-shrink_800_800/B56ZXx2G2CGoAc-/0/1743519248132?e=1776902400&v=beta&t=ptmB1rRwgM9MS-sBYorqTGsxmVa0FJIE9h86xHuJO9k""}","""AKLfR3WKvk""",true,"""Porter Olson""","{""US"",""en""}","""Porter""","""Olson""","""https://media.licdn.com/dms/im…","""0433769e922c7b83be73156ce3ad77…"
"""0433769e922c7b83be73156ce3ad77…","""6""","""Porter Olson""","""2026-04-07 22:42:20""","""2026-04-07 22:43:59""",null,99,5,"[{""2026-04-07 22:42:20"",""https://jobconnectionsproject.org/""}, {""2026-04-07 22:42:28"",""https://jobconnectionsproject.org/jobs/""}, … {""2026-04-07 22:43:57"",""https://jobconnectionsproject.org/jobs/pastry-chef-san-jose-ca-us/""}]","""128.187.116.26""","""26-04-07 10:43:59""","""9808807403""",0,"""https://jobconnectionsproject.…","""https://jobconnectionsproject.…","""https://www.linkedin.com/""","""128.187.116.26""","""128.187.116.26""","""4fc992883f07c878a88e2feaff047f…","""Mozilla/5.0 (Windows NT 10.0; …","""Desktop / Chrome / Windows""",true,false,"""2026-04-07 22:42:20""","""2026-04-07 22:43:59""",1,"""AKLfR3WKvk""","""porter.olson11@gmail.com""","""2026-04-07 22:42:20""","""2026-06-06 22:42:19""","[""128.187.116.26"", ""67.169.249.168""]","{""AKLfR3WKvk"",true,""Porter Olson"",{""US"",""en""},""Porter"",""Olson"",""porter.olson11@gmail.com"",""https://media.licdn.com/dms/image/v2/D5603AQG9oDGMQr7JTw/profile-displayphoto-shrink_800_800/B56ZXx2G2CGoAc-/0/1743519248132?e=1776902400&v=beta&t=ptmB1rRwgM9MS-sBYorqTGsxmVa0FJIE9h86xHuJO9k""}","""AKLfR3WKvk""",true,"""Porter Olson""","{""US"",""en""}","""Porter""","""Olson""","""https://media.licdn.com/dms/im…","""0433769e922c7b83be73156ce3ad77…"
"""1a551a87a2eecb9c80605fef06739f…","""8""","""Shark Frisbee""","""2026-04-07 22:41:54""","""2026-04-07 22:41:58""",null,4,2,"[{""2026-04-07 22:4

In [45]:
signed_in=merged.filter(pl.col('user_id').is_not_null())
signed_in

session_id,user_id,user_display_name,session_start,last_activity,session_end,duration_seconds,total_pageviews,visited_pages,ip,date,survey_id,treatment_group,post_url,job_ad_url,first_referrer,first_ip,last_ip,ip_hash,user_agent,device_summary,is_logged_in,login_state_changed,created_at,updated_at,record_id,linkedin_member_id,email,account_last_synced_at,linkedin_token_expires_at,linkedin_ip_history,linkedin_profile_data,sub,email_verified,name,locale,given_name,family_name,picture,session_survey_id
str,str,str,str,str,str,i64,i64,list[struct[2]],str,str,str,i64,str,str,str,str,str,str,str,str,bool,bool,str,str,i64,str,str,str,str,list[str],struct[8],str,bool,str,struct[2],str,str,str,str
"""0433769e922c7b83be73156ce3ad77…","""6""","""Porter Olson""","""2026-04-07 22:42:20""","""2026-04-07 22:43:59""",null,99,5,"[{""2026-04-07 22:42:20"",""https://jobconnectionsproject.org/""}, {""2026-04-07 22:42:28"",""https://jobconnectionsproject.org/jobs/""}, … {""2026-04-07 22:43:57"",""https://jobconnectionsproject.org/jobs/pastry-chef-san-jose-ca-us/""}]","""128.187.116.26""","""26-04-07 10:42:44""","""5256633647""",0,"""https://jobconnectionsproject.…","""https://jobconnectionsproject.…","""https://www.linkedin.com/""","""128.187.116.26""","""128.187.116.26""","""4fc992883f07c878a88e2feaff047f…","""Mozilla/5.0 (Windows NT 10.0; …","""Desktop / Chrome / Windows""",true,false,"""2026-04-07 22:42:20""","""2026-04-07 22:43:59""",1,"""AKLfR3WKvk""","""porter.olson11@gmail.com""","""2026-04-07 22:42:20""","""2026-06-06 22:42:19""","[""128.187.116.26"", ""67.169.249.168""]","{""AKLfR3WKvk"",true,""Porter Olson"",{""US"",""en""},""Porter"",""Olson"",""porter.olson11@gmail.com"",""https://media.licdn.com/dms/image/v2/D5603AQG9oDGMQr7JTw/profile-displayphoto-shrink_800_800/B56ZXx2G2CGoAc-/0/1743519248132?e=1776902400&v=beta&t=ptmB1rRwgM9MS-sBYorqTGsxmVa0FJIE9h86xHuJO9k""}","""AKLfR3WKvk""",true,"""Porter Olson""","{""US"",""en""}","""Porter""","""Olson""","""https://media.licdn.com/dms/im…","""0433769e922c7b83be73156ce3ad77…"
"""0433769e922c7b83be73156ce3ad77…","""6""","""Porter Olson""","""2026-04-07 22:42:20""","""2026-04-07 22:43:59""",null,99,5,"[{""2026-04-07 22:42:20"",""https://jobconnectionsproject.org/""}, {""2026-04-07 22:42:28"",""https://jobconnectionsproject.org/jobs/""}, … {""2026-04-07 22:43:57"",""https://jobconnectionsproject.org/jobs/pastry-chef-san-jose-ca-us/""}]","""128.187.116.26""","""26-04-07 10:43:59""","""9808807403""",0,"""https://jobconnectionsproject.…","""https://jobconnectionsproject.…","""https://www.linkedin.com/""","""128.187.116.26""","""128.187.116.26""","""4fc992883f07c878a88e2feaff047f…","""Mozilla/5.0 (Windows NT 10.0; …","""Desktop / Chrome / Windows""",true,false,"""2026-04-07 22:42:20""","""2026-04-07 22:43:59""",1,"""AKLfR3WKvk""","""porter.olson11@gmail.com""","""2026-04-07 22:42:20""","""2026-06-06 22:42:19""","[""128.187.116.26"", ""67.169.249.168""]","{""AKLfR3WKvk"",true,""Porter Olson"",{""US"",""en""},""Porter"",""Olson"",""porter.olson11@gmail.com"",""https://media.licdn.com/dms/image/v2/D5603AQG9oDGMQr7JTw/profile-displayphoto-shrink_800_800/B56ZXx2G2CGoAc-/0/1743519248132?e=1776902400&v=beta&t=ptmB1rRwgM9MS-sBYorqTGsxmVa0FJIE9h86xHuJO9k""}","""AKLfR3WKvk""",true,"""Porter Olson""","{""US"",""en""}","""Porter""","""Olson""","""https://media.licdn.com/dms/im…","""0433769e922c7b83be73156ce3ad77…"
"""1a551a87a2eecb9c80605fef06739f…","""8""","""Shark Frisbee""","""2026-04-07 22:41:54""","""2026-04-07 22:41:58""",null,4,2,"[{""2026-04-07 22:41:54"",""https://jobconnectionsproject.org/""}, {""2026-04-07 22:41:57"",""https://jobconnectionsproject.org/account/""}]",null,null,null,null,null,null,"""https://jobconnectionsproject.…","""128.187.116.26""","""128.187.116.26""","""4fc992883f07c878a88e2feaff047f…","""Mozilla/5.0 (Windows NT 10.0; …","""Desktop / Chrome / Windows""",true,false,"""2026-04-07 22:41:54""","""2026-04-07 22:41:58""",4,"""CrcsFvVv_j""","""sharkfrisbee@gmail.com""","""2026-04-07 22:4

In [46]:
signed_in_detail=signed_in.select(['given_name','family_name','picture']).filter(pl.col('given_name').is_not_null())
signed_in_detail=signed_in_detail.unique()
signed_in_detail

picture_link_list=signed_in_detail['picture'].to_list()

In [47]:
rows = []

for link in tqdm(picture_link_list):
    row = {
        "picture": link,

        "df_age": None,
        "df_face_confidence": None,
        "df_dominant_gender": None,
        "df_dominant_race": None,

        "df_prob_male": None,
        "df_prob_female": None,

        "df_prob_white": None,
        "df_prob_black": None,
        "df_prob_asian": None,
        "df_prob_indian": None,
        "df_prob_middle_eastern": None,
        "df_prob_latino": None,

        "df_status": None,
        "df_error": None,
    }

    try:
        img_data = requests.get(link, timeout=20).content

        with open("temp.jpg", "wb") as f:
            f.write(img_data)

        result = DeepFace.analyze(
            img_path="temp.jpg",
            actions=["age", "gender", "race"],
            enforce_detection=True
        )

        face = result[0]

        row["df_age"] = face.get("age")
        row["df_face_confidence"] = face.get("face_confidence")
        row["df_dominant_gender"] = face.get("dominant_gender")
        row["df_dominant_race"] = face.get("dominant_race")

        gender_dict = face.get("gender", {})
        race_dict = face.get("race", {})

        row["df_prob_male"] = float(gender_dict["Man"]) if "Man" in gender_dict else None
        row["df_prob_female"] = float(gender_dict["Woman"]) if "Woman" in gender_dict else None

        row["df_prob_white"] = float(race_dict["white"]) if "white" in race_dict else None
        row["df_prob_black"] = float(race_dict["black"]) if "black" in race_dict else None
        row["df_prob_asian"] = float(race_dict["asian"]) if "asian" in race_dict else None
        row["df_prob_indian"] = float(race_dict["indian"]) if "indian" in race_dict else None
        row["df_prob_middle_eastern"] = float(race_dict["middle eastern"]) if "middle eastern" in race_dict else None
        row["df_prob_latino"] = float(race_dict["latino hispanic"]) if "latino hispanic" in race_dict else None

        row["df_status"] = "ok"

    except Exception as e:
        msg = str(e)

        if "Face could not be detected" in msg:
            row["df_status"] = "no_face_detected"
        else:
            row["df_status"] = "error"

        row["df_error"] = msg

    rows.append(row)

pic_df = pl.DataFrame(rows)

pic_df

100%|██████████| 4/4 [00:03<00:00,  1.07it/s]


picture,df_age,df_face_confidence,df_dominant_gender,df_dominant_race,df_prob_male,df_prob_female,df_prob_white,df_prob_black,df_prob_asian,df_prob_indian,df_prob_middle_eastern,df_prob_latino,df_status,df_error
str,i64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str
"""https://media.licdn.com/dms/im…",null,null,null,null,null,null,null,null,null,null,null,null,"""no_face_detected""","""Face could not be detected in …"
"""https://media.licdn.com/dms/im…",23,0.92,"""Man""","""white""",98.778412,1.221596,68.907288,0.085356,1.472363,0.598139,13.370648,15.566208,"""ok""",null
"""https://media.licdn.com/dms/im…",25,0.92,"""Man""","""white""",99.571747,0.42825,94.859116,0.000324,0.029293,0.011143,2.819904,2.280221,"""ok""",null
"""https://media.licdn.com/dms/im…",32,0.9,"""Man""","""white""",99.723633,0.276374,98.395454,0.000001,0.000003,0.000235,1.036455,0.567853,"""ok""",null


In [48]:
import polars as pl
import pandas as pd
import gender_guesser.detector as gender
from ethnicolr import pred_fl_reg_name

d = gender.Detector(case_sensitive=False)

rows = []

for row in signed_in_detail.iter_rows(named=True):
    first_name = row["given_name"]
    last_name = row["family_name"]

    # gender-guesser
    gender_guess = d.get_gender(first_name) if first_name is not None else None

    temp_df = pd.DataFrame({
        "first_name": [first_name],
        "last_name": [last_name]
    })

    try:
        result = pred_fl_reg_name(temp_df, "last_name", "first_name")
        r = result.iloc[0]

        rows.append({
            "given_name": first_name,
            "family_name": last_name,

            # gender-guesser output
            "gg_gender_label": gender_guess,

            # ethnicolr output
            "eth_asian": r.get("asian"),
            "eth_hispanic": r.get("hispanic"),
            "eth_nh_black": r.get("nh_black"),
            "eth_nh_white": r.get("nh_white"),
            "eth_race_label": r.get("race"),
            "eth_processing_status": r.get("processing_status"),

            "ethnicolr_status": "ok"
        })

    except Exception as e:
        rows.append({
            "given_name": first_name,
            "family_name": last_name,

            "gg_gender_label": gender_guess,

            "eth_asian": None,
            "eth_hispanic": None,
            "eth_nh_black": None,
            "eth_nh_white": None,
            "eth_race_label": None,
            "eth_processing_status": None,

            "ethnicolr_status": f"error: {str(e)}"
        })

ethnicolr_gender_df = pl.DataFrame(rows)

ethnicolr_gender_df

2026-04-07 21:31:12,437 - INFO - Processing 1 full names
2026-04-07 21:31:12,437 - INFO - Applying Florida voter name model to 1 processable names (confidence interval: 1.0)
2026-04-07 21:31:12,437 - INFO - Data filtering summary: 1 → 1 rows (kept 100.0%)
2026-04-07 21:31:12,552 - INFO - Successfully predicted 1 of 1 names (100.0%)
2026-04-07 21:31:12,552 - INFO - Added columns: name_normalized, __name, hispanic, race, nh_white, name_normalized_clean, nh_black, asian, processing_status
2026-04-07 21:31:12,557 - INFO - Processing 1 full names
2026-04-07 21:31:12,557 - INFO - Applying Florida voter name model to 1 processable names (confidence interval: 1.0)
2026-04-07 21:31:12,557 - INFO - Data filtering summary: 1 → 1 rows (kept 100.0%)
2026-04-07 21:31:12,678 - INFO - Successfully predicted 1 of 1 names (100.0%)
2026-04-07 21:31:12,678 - INFO - Added columns: name_normalized, __name, hispanic, race, nh_white, name_normalized_clean, nh_black, asian, processing_status
2026-04-07 21:31:1

given_name,family_name,gg_gender_label,eth_asian,eth_hispanic,eth_nh_black,eth_nh_white,eth_race_label,eth_processing_status,ethnicolr_status
str,str,str,f64,f64,f64,f64,str,str,str
"""Shark""","""Frisbee""","""unknown""",0.001475,0.005477,0.0429,0.950148,"""nh_white""","""processed""","""ok"""
"""Spencer""","""Bailey""","""male""",0.003404,0.014741,0.130191,0.851664,"""nh_white""","""processed""","""ok"""
"""Porter""","""Olson""","""male""",0.000641,0.00363,0.007302,0.988427,"""nh_white""","""processed""","""ok"""
"""Tanner""","""Eastmond""","""male""",0.0021,0.012004,0.117149,0.868747,"""nh_white""","""processed""","""ok"""


In [49]:
merged=merged.join(pic_df, on=['picture'],how='left')

In [50]:
merged=merged.join(ethnicolr_gender_df,on=['given_name','family_name'])


In [51]:
merged

session_id,user_id,user_display_name,session_start,last_activity,session_end,duration_seconds,total_pageviews,visited_pages,ip,date,survey_id,treatment_group,post_url,job_ad_url,first_referrer,first_ip,last_ip,ip_hash,user_agent,device_summary,is_logged_in,login_state_changed,created_at,updated_at,record_id,linkedin_member_id,email,account_last_synced_at,linkedin_token_expires_at,linkedin_ip_history,linkedin_profile_data,sub,email_verified,name,locale,given_name,family_name,picture,session_survey_id,df_age,df_face_confidence,df_dominant_gender,df_dominant_race,df_prob_male,df_prob_female,df_prob_white,df_prob_black,df_prob_asian,df_prob_indian,df_prob_middle_eastern,df_prob_latino,df_status,df_error,gg_gender_label,eth_asian,eth_hispanic,eth_nh_black,eth_nh_white,eth_race_label,eth_processing_status,ethnicolr_status
str,str,str,str,str,str,i64,i64,list[struct[2]],str,str,str,i64,str,str,str,str,str,str,str,str,bool,bool,str,str,i64,str,str,str,str,list[str],struct[8],str,bool,str,struct[2],str,str,str,str,i64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,f64,f64,f64,f64,str,str,str
"""0433769e922c7b83be73156ce3ad77…","""6""","""Porter Olson""","""2026-04-07 22:42:20""","""2026-04-07 22:43:59""",null,99,5,"[{""2026-04-07 22:42:20"",""https://jobconnectionsproject.org/""}, {""2026-04-07 22:42:28"",""https://jobconnectionsproject.org/jobs/""}, … {""2026-04-07 22:43:57"",""https://jobconnectionsproject.org/jobs/pastry-chef-san-jose-ca-us/""}]","""128.187.116.26""","""26-04-07 10:42:44""","""5256633647""",0,"""https://jobconnectionsproject.…","""https://jobconnectionsproject.…","""https://www.linkedin.com/""","""128.187.116.26""","""128.187.116.26""","""4fc992883f07c878a88e2feaff047f…","""Mozilla/5.0 (Windows NT 10.0; …","""Desktop / Chrome / Windows""",true,false,"""2026-04-07 22:42:20""","""2026-04-07 22:43:59""",1,"""AKLfR3WKvk""","""porter.olson11@gmail.com""","""2026-04-07 22:42:20""","""2026-06-06 22:42:19""","[""128.187.116.26"", ""67.169.249.168""]","{""AKLfR3WKvk"",true,""Porter Olson"",{""US"",""en""},""Porter"",""Olson"",""porter.olson11@gmail.com"",""https://media.licdn.com/dms/image/v2/D5603AQG9oDGMQr7JTw/profile-displayphoto-shrink_800_800/B56ZXx2G2CGoAc-/0/1743519248132?e=1776902400&v=beta&t=ptmB1rRwgM9MS-sBYorqTGsxmVa0FJIE9h86xHuJO9k""}","""AKLfR3WKvk""",true,"""Porter Olson""","{""US"",""en""}","""Porter""","""Olson""","""https://media.licdn.com/dms/im…","""0433769e922c7b83be73156ce3ad77…",25,0.92,"""Man""","""white""",99.571747,0.42825,94.859116,0.000324,0.029293,0.011143,2.819904,2.280221,"""ok""",null,"""male""",0.000641,0.00363,0.007302,0.988427,"""nh_white""","""processed""","""ok"""
"""0433769e922c7b83be73156ce3ad77…","""6""","""Porter Olson""","""2026-04-07 22:42:20""","""2026-04-07 22:43:59""",null,99,5,"[{""2026-04-07 22:42:20"",""https://jobconnectionsproject.org/""}, {""2026-04-07 22:42:28"",""https://jobconnectionsproject.org/jobs/""}, … {""2026-04-07 22:43:57"",""https://jobconnectionsproject.org/jobs/pastry-chef-san-jose-ca-us/""}]","""128.187.116.26""","""26-04-07 10:43:59""","""9808807403""",0,"""https://jobconnectionsproject.…","""https://jobconnectionsproject.…","""https://www.linkedin.com/""","""128.187.116.26""","""128.187.116.26""","""4fc992883f07c878a88e2feaff047f…","""Mozilla/5.0 (Windows NT 10.0; …","""Desktop / Chrome / Windows""",true,false,"""2026-04-07 22:42:20""","""2026-04-07 22:43:59""",1,"""AKLfR3WKvk""","""porter.olson11@gmail.com""","""2026-04-07 22:42:20""","""2026-06-06 22:42:19""","[""128.187.116.26"", ""67.169.249.168""]","{""AKLfR3WKvk"",true,""Porter Olson"",{""US"",""en""},""Porter"",""Olson"",""porter.olson11@gmail.com"",""https://media.licdn.com/dms/image/v2/D5603AQG9oDGMQr7JTw/profile-displayphoto-shrink_800_800/B56ZXx2G2CGoAc-/0/1743519248132?e=1776902400&v=beta&t=ptmB1rRwgM9MS-sBYorqTGsxmVa0FJIE9h86xHuJO9k""}","""AKLfR3WKvk""",true,"""Porter Olson""","{""US"",""en""}","""Porter""","""Olson""","""https://media.licdn.com/dms/im…","""0433769e922c7b83be73156ce3ad7